In [1]:
# Install requirements if running in Kaggle
!pip install lpips pytorch-msssim scikit-image torchmetrics torch-fidelity --quiet

import os
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision.transforms as T
import torch.nn.functional as F
import matplotlib.pyplot as plt
import random
import time
import csv
import copy
from skimage.metrics import peak_signal_noise_ratio, structural_similarity
from torchmetrics.image.fid import FrechetInceptionDistance
from collections import defaultdict

# =================== Config ===================
class Config:
    batch_size = 4
    epochs = 100
    patience = 30
    lr = 1e-3
    attention = 'gate' # options: none, gate, light
    seeds = [0, 1, 2]
    base_dir = '/kaggle/input/rice-remote-sensing-images-for-cloud-removal/RICE/RICE1'
    checkpoints_dir = '/kaggle/working/checkpoints'
    log_file = 'training_log.csv'
    image_size = (256, 256)
    split_ratio = 0.9 # 90/10 train/val split

# =================== Reproducibility ===================
def set_seed(seed=42):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# =================== Dataset Split ===================
def split_dataset(cloudy_dir, clear_dir, split_ratio=0.9):
    filenames = [f for f in os.listdir(cloudy_dir) if f.endswith('.jpg') or f.endswith('.png')]
    filenames.sort()
    # Pairwise shuffle
    indices = np.arange(len(filenames))
    np.random.shuffle(indices)
    split_idx = int(len(indices) * split_ratio)
    train_indices = indices[:split_idx]
    val_indices = indices[split_idx:]
    train_files = [filenames[i] for i in train_indices]
    val_files = [filenames[i] for i in val_indices]
    return train_files, val_files

# =================== Dataset ===================
class RICEDataset(Dataset):
    def __init__(self, cloudy_dir, clear_dir, filenames, transform=None):
        self.cloudy_dir = cloudy_dir
        self.clear_dir = clear_dir
        self.filenames = filenames
        self.transform = transform
    def __len__(self):
        return len(self.filenames)
    def __getitem__(self, idx):
        fname = self.filenames[idx]
        cloudy = Image.open(os.path.join(self.cloudy_dir, fname)).convert('RGB')
        clear = Image.open(os.path.join(self.clear_dir, fname)).convert('RGB')
        if self.transform:
            seed = idx
            random.seed(seed)
            torch.manual_seed(seed)
            cloudy = self.transform(cloudy)
            random.seed(seed)
            torch.manual_seed(seed)
            clear = self.transform(clear)
        return cloudy, clear, fname

# =================== Attention Modules ===================
class AttentionGate(nn.Module):
    def __init__(self, F_g, F_l, F_int):
        super().__init__()
        self.W_g = nn.Sequential(
            nn.Conv2d(F_g, F_int, kernel_size=1, stride=1, padding=0, bias=True),
            nn.GroupNorm(num_groups=min(8, F_int), num_channels=F_int)
        )
        self.W_x = nn.Sequential(
            nn.Conv2d(F_l, F_int, kernel_size=1, stride=1, padding=0, bias=True),
            nn.GroupNorm(num_groups=min(8, F_int), num_channels=F_int)
        )
        self.psi = nn.Sequential(
            nn.Conv2d(F_int, 1, kernel_size=1, stride=1, padding=0, bias=True),
            nn.GroupNorm(num_groups=1, num_channels=1),
            nn.Sigmoid()
        )
        self.relu = nn.ReLU(inplace=True)
    def forward(self, g, x):
        g1 = self.W_g(g)
        x1 = self.W_x(x)
        if g1.shape[2:] != x1.shape[2:]:
            g1 = F.interpolate(g1, size=x1.shape[2:], mode='bilinear', align_corners=False)
        psi = self.relu(g1 + x1)
        psi = self.psi(psi)
        return x * psi

class LightAttention(nn.Module):
    def __init__(self, F_g, F_l, F_int):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Conv2d(F_g+F_l, F_int, 1),
            nn.ReLU(inplace=True),
            nn.Conv2d(F_int, 1, 1),
            nn.Sigmoid()
        )
    def forward(self, g, x):
        # Concatenate and apply a lighter attention
        x1 = torch.cat([g, x], dim=1)
        attn = self.fc(x1)
        return x * attn

# =================== Attention U-Net ===================
class AttentionIdentity(nn.Module):
    def forward(self, g, x):
        return x

class AttUNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=3, attention_mode='gate'):
        super().__init__()
        def CBR(in_ch, out_ch):
            return nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 3, padding=1),
                nn.GroupNorm(num_groups=min(8, out_ch), num_channels=out_ch),
                nn.ReLU(inplace=True)
            )
        self.enc1 = CBR(in_channels, 32)
        self.enc2 = CBR(32, 64)
        self.enc3 = CBR(64, 128)
        self.enc4 = CBR(128, 256)
        self.pool = nn.MaxPool2d(2)
        self.bottleneck = CBR(256, 512)
        self.attention_mode = attention_mode
        if attention_mode == 'none':
            self.att4 = AttentionIdentity()
            self.att3 = AttentionIdentity()
            self.att2 = AttentionIdentity()
            self.att1 = AttentionIdentity()
        elif attention_mode == 'gate':
            self.att4 = AttentionGate(F_g=256, F_l=256, F_int=128)
            self.att3 = AttentionGate(F_g=128, F_l=128, F_int=64)
            self.att2 = AttentionGate(F_g=64, F_l=64, F_int=32)
            self.att1 = AttentionGate(F_g=32, F_l=32, F_int=16)
        elif attention_mode == 'light':
            self.att4 = LightAttention(F_g=256, F_l=256, F_int=64)
            self.att3 = LightAttention(F_g=128, F_l=128, F_int=32)
            self.att2 = LightAttention(F_g=64, F_l=64, F_int=16)
            self.att1 = LightAttention(F_g=32, F_l=32, F_int=8)
        else:
            raise ValueError(f"Unknown attention mode: {attention_mode}")
        
        self.up4 = nn.ConvTranspose2d(512, 256, 2, stride=2)
        self.dec4 = CBR(512, 256)
        self.up3 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.dec3 = CBR(256, 128)
        self.up2 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.dec2 = CBR(128, 64)
        self.up1 = nn.ConvTranspose2d(64, 32, 2, stride=2)
        self.dec1 = CBR(64, 32)
        self.final = nn.Conv2d(32, out_channels, 1)
    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))
        b = self.bottleneck(self.pool(e4))
        d4 = self.up4(b)
        a4 = self.att4(g=d4, x=e4)
        if d4.shape[2:] != a4.shape[2:]:
            d4 = F.interpolate(d4, size=a4.shape[2:], mode='bilinear', align_corners=False)
        d4 = torch.cat([d4, a4], dim=1)
        d4 = self.dec4(d4)
        d3 = self.up3(d4)
        a3 = self.att3(g=d3, x=e3)
        if d3.shape[2:] != a3.shape[2:]:
            d3 = F.interpolate(d3, size=a3.shape[2:], mode='bilinear', align_corners=False)
        d3 = torch.cat([d3, a3], dim=1)
        d3 = self.dec3(d3)
        d2 = self.up2(d3)
        a2 = self.att2(g=d2, x=e2)
        if d2.shape[2:] != a2.shape[2:]:
            d2 = F.interpolate(d2, size=a2.shape[2:], mode='bilinear', align_corners=False)
        d2 = torch.cat([d2, a2], dim=1)
        d2 = self.dec2(d2)
        d1 = self.up1(d2)
        a1 = self.att1(g=d1, x=e1)
        if d1.shape[2:] != a1.shape[2:]:
            d1 = F.interpolate(d1, size=a1.shape[2:], mode='bilinear', align_corners=False)
        d1 = torch.cat([d1, a1], dim=1)
        d1 = self.dec1(d1)
        out = self.final(d1)
        out = (out + x).clamp(0, 1)
        return out

# =================== Loss & Metrics ===================
class PSNRLoss(nn.Module):
    def __init__(self, mse_weight=1.0, l1_weight=0.2):
        super().__init__()
        self.l1 = nn.L1Loss()
        self.mse = nn.MSELoss()
        self.mse_weight = mse_weight
        self.l1_weight = l1_weight
    def forward(self, output, target):
        return self.mse_weight * self.mse(output, target) + self.l1_weight * self.l1(output, target)

def compute_psnr(pred, gt):
    pred = np.clip(pred, 0, 1)
    gt = np.clip(gt, 0, 1)
    return peak_signal_noise_ratio(gt, pred, data_range=1)
def compute_ssim(pred, gt):
    pred = np.clip(pred, 0, 1)
    gt = np.clip(gt, 0, 1)
    return structural_similarity(gt, pred, channel_axis=-1, data_range=1)
def compute_mae(pred, gt):
    return np.mean(np.abs(pred - gt))
def compute_mse(pred, gt):
    return np.mean((pred - gt) ** 2)

# =================== Logging ===================
class TrainingLogger:
    def __init__(self, file_path, fieldnames):
        self.file_path = file_path
        self.fieldnames = fieldnames
        with open(self.file_path, 'w', newline='') as f:
            writer = csv.DictWriter(f, fieldnames=self.fieldnames)
            writer.writeheader()
    def log(self, row_dict):
        with open(self.file_path, 'a', newline='') as f:
            writer = csv.DictWriter(f, fieldnames=self.fieldnames)
            writer.writerow(row_dict)

# =================== Evaluation ===================
def evaluate(model, loader, device):
    model.eval()
    psnr_list, ssim_list, mae_list, mse_list, infer_times = [], [], [], [], []
    with torch.no_grad():
        for cloudy, clear, _ in loader:
            cloudy, clear = cloudy.to(device), clear.to(device)
            start = time.time()
            pred = model(cloudy)
            infer_times.append((time.time() - start) / pred.size(0))
            pred_np = pred.cpu().numpy()
            clear_np = clear.cpu().numpy()
            for i in range(pred_np.shape[0]):
                p = np.transpose(pred_np[i], (1,2,0))
                t = np.transpose(clear_np[i], (1,2,0))
                psnr_list.append(compute_psnr(p, t))
                ssim_list.append(compute_ssim(p, t))
                mae_list.append(compute_mae(p, t))
                mse_list.append(compute_mse(p, t))
    metrics = {
        'PSNR': np.mean(psnr_list),
        'SSIM': np.mean(ssim_list),
        'MAE': np.mean(mae_list),
        'MSE': np.mean(mse_list),
        'InferenceTime': np.mean(infer_times)
    }
    return metrics

# =================== Triptych Panel ===================
def make_triptych_panel(model, loader, device, save_path, n=6):
    model.eval()
    cloudy, clear, fnames = next(iter(loader))
    batch_size = cloudy.shape[0]
    plot_n = min(n, batch_size)
    with torch.no_grad():
        pred = model(cloudy.to(device)).cpu()
    plt.figure(figsize=(12, plot_n*2))
    for i in range(plot_n):
        plt.subplot(plot_n, 3, i*3 + 1)
        plt.imshow(cloudy[i].permute(1,2,0).numpy())
        plt.title('Cloudy')
        plt.axis('off')
        plt.subplot(plot_n, 3, i*3 + 2)
        plt.imshow(pred[i].permute(1,2,0).clamp(0,1).numpy())
        plt.title('Predicted')
        plt.axis('off')
        plt.subplot(plot_n, 3, i*3 + 3)
        plt.imshow(clear[i].permute(1,2,0).numpy())
        plt.title('Clear')
        plt.axis('off')
    plt.tight_layout()
    plt.savefig(save_path, bbox_inches='tight')
    plt.close()

# =================== Main Multi-Seed Loop ===================
def main(config):
    os.makedirs(config.checkpoints_dir, exist_ok=True)
    all_metrics = defaultdict(list)
    logger = TrainingLogger(config.log_file, fieldnames=['seed', 'epoch', 'train_loss', 'val_psnr', 'val_ssim', 'val_mae', 'val_mse', 'val_infer_time', 'lr'])

    # Get train/val split once per run, but shuffle for each seed
    cloudy_dir = os.path.join(config.base_dir, 'cloud')
    clear_dir = os.path.join(config.base_dir, 'label')
    all_filenames = [f for f in os.listdir(cloudy_dir) if f.endswith('.jpg') or f.endswith('.png')]
    all_filenames.sort()
    split_idx = int(len(all_filenames) * config.split_ratio)
    
    for seed in config.seeds:
        set_seed(seed)
        indices = np.arange(len(all_filenames))
        np.random.shuffle(indices)
        train_indices = indices[:split_idx]
        val_indices = indices[split_idx:]
        train_files = [all_filenames[i] for i in train_indices]
        val_files = [all_filenames[i] for i in val_indices]

        train_transform = T.Compose([
            T.Resize(config.image_size),
            T.ToTensor()
        ])
        test_transform = T.Compose([
            T.Resize(config.image_size),
            T.ToTensor()
        ])
        train_ds = RICEDataset(cloudy_dir, clear_dir, train_files, transform=train_transform)
        val_ds = RICEDataset(cloudy_dir, clear_dir, val_files, transform=test_transform)
        train_dl = DataLoader(train_ds, batch_size=config.batch_size, shuffle=True, num_workers=2, worker_init_fn=lambda wid: set_seed(seed+wid), pin_memory=True)
        val_dl = DataLoader(val_ds, batch_size=config.batch_size, shuffle=False, num_workers=2, worker_init_fn=lambda wid: set_seed(seed+wid), pin_memory=True)
        
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        model = AttUNet(attention_mode=config.attention).to(device)
        criterion = PSNRLoss(mse_weight=1.0, l1_weight=0.2)
        optimizer = torch.optim.Adam(model.parameters(), lr=config.lr, betas=(0.9, 0.999))
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', patience=10, factor=0.5, verbose=True)
        best_psnr = -1
        patience_counter = 0
        best_model_wts = copy.deepcopy(model.state_dict())
        scaler = torch.cuda.amp.GradScaler()

        for epoch in range(1, config.epochs+1):
            model.train()
            total_loss = 0.0
            for cloudy, clear, _ in train_dl:
                cloudy, clear = cloudy.to(device), clear.to(device)
                optimizer.zero_grad()
                with torch.cuda.amp.autocast():
                    pred = model(cloudy)
                    loss = criterion(pred, clear)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
                total_loss += loss.item() * cloudy.size(0)
            train_loss = total_loss / len(train_dl.dataset)
            torch.cuda.empty_cache()
            val_metrics = evaluate(model, val_dl, device)
            scheduler.step(val_metrics['PSNR'])
            lr_now = optimizer.param_groups[0]['lr']
            logger.log({
                'seed': seed,
                'epoch': epoch,
                'train_loss': train_loss,
                'val_psnr': val_metrics['PSNR'],
                'val_ssim': val_metrics['SSIM'],
                'val_mae': val_metrics['MAE'],
                'val_mse': val_metrics['MSE'],
                'val_infer_time': val_metrics['InferenceTime'],
                'lr': lr_now
            })
            print(f"[Seed {seed}] Epoch {epoch}/{config.epochs} | Train Loss: {train_loss:.4f} | Val PSNR: {val_metrics['PSNR']:.2f} | Val SSIM: {val_metrics['SSIM']:.4f} | MAE: {val_metrics['MAE']:.4f} | MSE: {val_metrics['MSE']:.4f} | InferTime: {val_metrics['InferenceTime']*1000:.2f} ms | LR: {lr_now:.6f}")
            if val_metrics['PSNR'] > best_psnr:
                best_psnr = val_metrics['PSNR']
                patience_counter = 0
                best_model_wts = copy.deepcopy(model.state_dict())
                torch.save(model.state_dict(), f"{config.checkpoints_dir}/best_model_seed{seed}_att{config.attention}.pt")
            else:
                patience_counter += 1
            if patience_counter >= config.patience:
                print(f"[Seed {seed}] Early stopping triggered.")
                break
            if epoch % 5 == 0 or epoch == 1:
                make_triptych_panel(model, val_dl, device, f"{config.checkpoints_dir}/triptych_seed{seed}_epoch{epoch}.png", n=6)
        model.load_state_dict(best_model_wts)
        print(f"[Seed {seed}] Best Validation PSNR: {best_psnr:.4f}")
        make_triptych_panel(model, val_dl, device, f"{config.checkpoints_dir}/final_triptych_seed{seed}.png", n=6)
        # Evaluate test set if available (or re-use val as test)
        test_metrics = evaluate(model, val_dl, device)
        for k, v in test_metrics.items():
            all_metrics[k].append(v)

    print("\n==== Final Metrics (mean ± std) ====")
    for k, v in all_metrics.items():
        arr = np.array(v)
        print(f"{k}: {arr.mean():.4f} ± {arr.std():.4f}")

if __name__ == '__main__':
    attention_modes = ['none', 'light', 'gate']
    for att_mode in attention_modes:
        print(f"\n=== Running ablation: {att_mode} attention ===")
        class AblationConfig(Config):
            attention = att_mode
            log_file = f"training_log_{att_mode}.csv"
            checkpoints_dir = f"/kaggle/working/checkpoints_{att_mode}"
        os.makedirs(AblationConfig.checkpoints_dir, exist_ok=True)
        main(AblationConfig)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.8/53.8 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 90.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 76.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 33.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 29.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 77.1 MB/s eta 0:00:00

=== Running ablation: none attention ===


/tmp/ipykernel_19/864444970.py:340: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
/tmp/ipykernel_19/864444970.py:348: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


[Seed 0] Epoch 1/100 | Train Loss: 0.0389 | Val PSNR: 22.38 | Val SSIM: 0.8518 | MAE: 0.0769 | MSE: 0.0097 | InferTime: 3.07 ms | LR: 0.001000
[Seed 0] Epoch 2/100 | Train Loss: 0.0236 | Val PSNR: 23.26 | Val SSIM: 0.8186 | MAE: 0.0648 | MSE: 0.0078 | InferTime: 1.35 ms | LR: 0.001000
[Seed 0] Epoch 3/100 | Train Loss: 0.0168 | Val PSNR: 23.19 | Val SSIM: 0.8709 | MAE: 0.0712 | MSE: 0.0092 | InferTime: 1.22 ms | LR: 0.001000
[Seed 0] Epoch 4/100 | Train Loss: 0.0138 | Val PSNR: 27.12 | Val SSIM: 0.9233 | MAE: 0.0427 | MSE: 0.0036 | InferTime: 1.52 ms | LR: 0.001000
[Seed 0] Epoch 5/100 | Train Loss: 0.0117 | Val PSNR: 27.32 | Val SSIM: 0.9322 | MAE: 0.0430 | MSE: 0.0037 | InferTime: 1.32 ms | LR: 0.001000
[Seed 0] Epoch 6/100 | Train Loss: 0.0136 | Val PSNR: 28.96 | Val SSIM: 0.9314 | MAE: 0.0342 | MSE: 0.0027 | InferTime: 1.42 ms | LR: 0.001000
[Seed 0] Epoch 7/100 | Train Loss: 0.0110 | Val PSNR: 27.37 | Val SSIM: 0.9427 | MAE: 0.0419 | MSE: 0.0033 | InferTime: 1.27 ms | LR: 0.001000